In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

from ai_agents.agents.email_finder import run_batch, SourceType
from google_utils.google_sheet import GoogleSheetService

from dotenv import load_dotenv
load_dotenv()


CHECKPOINT_FILE = "email_finder_checkpoint.json"
BATCH_SIZE = 50
CONCURRENCY = 5

In [ ]:
def load_checkpoint(spreadsheet_id: str) -> dict:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
            # Only resume if same spreadsheet
            if data.get("spreadsheet_id") == spreadsheet_id:
                print(f"Resuming from checkpoint — {len(data['processed_indices'])} rows already processed")
                return data
    return {
        "spreadsheet_id": spreadsheet_id,
        "processed_indices": [],
        "found_count": 0,
        "not_found_count": 0,
        "failed_count": 0,
        "last_updated": None,
    }


def save_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_updated"] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)


def clear_checkpoint() -> None:
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared")

In [ ]:
def result_to_sheet_row(raw_row: dict, result: dict) -> list:
    """Convert a processed result into a row for Email_Finder sheet."""
    best_email = result.get("best_email") or {}
    if isinstance(best_email, dict):
        email = best_email.get("email", "")
        source = best_email.get("source", "")
        confidence = best_email.get("confidence", "")
    else:
        email = ""
        source = ""
        confidence = ""

    return [
        raw_row.get("Podcast Name", ""),
        raw_row.get("Podcast Website", ""),
        email,
        source,
        str(confidence),
        result.get("status", ""),
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ]

In [ ]:
def run_email_finder_pipeline(
    spreadsheet_id: str,
    source_type: SourceType,
    needs_enrichment_sheet: str = "Needs_enrichment",
    email_finder_sheet: str = "Email_Finder",
    batch_size: int = BATCH_SIZE,
    concurrency: int = CONCURRENCY,
    dry_run: bool = False,
) -> dict:
    """
    Main pipeline entry point.

    Args:
        spreadsheet_id: Google Sheet ID
        source_type: SourceType enum value
        needs_enrichment_sheet: Sheet to read from
        email_finder_sheet: Sheet to write found emails to
        batch_size: Rows per batch
        concurrency: Concurrent leads per batch
        dry_run: If True, don't write to sheets (for testing)
    """
    sheet_service = GoogleSheetService()
    checkpoint = load_checkpoint(spreadsheet_id)
    already_processed = set(checkpoint["processed_indices"])

    # ── Read all rows ──
    print(f"Reading {needs_enrichment_sheet}...")
    success, df = sheet_service.get_sheet_data(spreadsheet_id, needs_enrichment_sheet)
    assert success, f"Failed to read sheet: {df}"

    total_rows = len(df)
    print(f"Total rows: {total_rows} | Already processed: {len(already_processed)}")

    # Filter out already processed rows
    remaining_indices = [i for i in range(total_rows) if i not in already_processed]
    print(f"Remaining to process: {len(remaining_indices)}")

    if not remaining_indices:
        print("Nothing to process — all rows already handled")
        return checkpoint

    # Ensure Email_Finder sheet has headers if empty
    if not dry_run:
        _ensure_email_finder_headers(sheet_service, spreadsheet_id, email_finder_sheet)

    # ── Process in batches ──
    all_processed_indices = []

    for batch_start in range(0, len(remaining_indices), batch_size):
        batch_indices = remaining_indices[batch_start: batch_start + batch_size]
        batch_rows = [df.iloc[i].to_dict() for i in batch_indices]

        print(f"\nBatch {batch_start // batch_size + 1} — rows {batch_indices[0]+1} to {batch_indices[-1]+1}")

        # Run batch through email finder
        results = run_batch(
            rows=batch_rows,
            source_type=source_type,
            concurrency=concurrency,
        )

        # Separate found vs not found
        found_rows = []
        for i, (row_idx, raw_row, result) in enumerate(
            zip(batch_indices, batch_rows, results)
        ):
            status = result.get("status", "")

            if status == "email_found":
                found_rows.append(result_to_sheet_row(raw_row, result))
                checkpoint["found_count"] += 1
            elif status == "failed":
                checkpoint["failed_count"] += 1
            else:
                checkpoint["not_found_count"] += 1

            all_processed_indices.append(row_idx)
            checkpoint["processed_indices"].append(row_idx)

        # Write found emails to Email_Finder sheet
        if found_rows and not dry_run:
            success, msg = sheet_service.append_rows(
                spreadsheet_id, email_finder_sheet, found_rows
            )
            print(f"  → Wrote {len(found_rows)} emails to {email_finder_sheet}: {msg}")

        # Save checkpoint after every batch
        save_checkpoint(checkpoint)
        print(f"  → Checkpoint saved | Found: {checkpoint['found_count']} | Not found: {checkpoint['not_found_count']} | Failed: {checkpoint['failed_count']}")

    # ── Remove processed rows from Needs_Enrichment ──
    if all_processed_indices and not dry_run:
        print(f"\nRemoving {len(all_processed_indices)} processed rows from {needs_enrichment_sheet}...")
        success, msg = sheet_service.delete_rows_by_indices(
            spreadsheet_id,
            needs_enrichment_sheet,
            all_processed_indices,
        )
        print(f"  → {msg}")

    # ── Final summary ──
    print(f"\n✓ Pipeline complete")
    print(f"  Found:     {checkpoint['found_count']}")
    print(f"  Not found: {checkpoint['not_found_count']}")
    print(f"  Failed:    {checkpoint['failed_count']}")

    if not dry_run:
        clear_checkpoint()

    return checkpoint


def _ensure_email_finder_headers(
    sheet_service: GoogleSheetService,
    spreadsheet_id: str,
    sheet_name: str,
) -> None:
    """Add headers to Email_Finder sheet if it's empty."""
    success, values = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A1:G1")
    if not success or not values:
        headers = [[
            "Podcast Name",
            "Podcast Website",
            "Email Found",
            "Email Source",
            "Confidence",
            "Status",
            "Processed At",
        ]]
        sheet_service.append_rows(spreadsheet_id, sheet_name, headers)
        print(f"Added headers to {sheet_name}")

In [ ]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1SSR4yXUpMqsmFpWIV27gt6qfmLdtOMVjgmsWycIabUo/edit?gid=806037148#gid=806037148"
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

result = run_email_finder_pipeline(
    spreadsheet_id=spreadsheet_id,
    source_type=SourceType.PODSCAN_HOST,
    dry_run=True,
    batch_size=5,
    concurrency=2,
)